In [1]:
!pip install "transformers<5" torch sentencepiece
! pip install youtube-transcript-api
! pip install torch
!pip install faiss-gpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━

In [ ]:
from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import faiss

In [96]:
qwen_model_name  = "microsoft/Phi-4-mini-instruct"
qwen_tokenizer  = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model  = AutoModelForCausalLM.from_pretrained(qwen_model_name, torch_dtype=torch.float16, device_map="auto")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

: 

: 

: 

In [78]:
def extract_video_id(url: str) -> str:
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    video_ids = qs.get('v')

    if not video_ids:
        raise ValueError(f"No video id found in URL: {url}")
    return video_ids[0]

urllink = input("Enter Url of the Video :\n")
if __name__ == "__main__":
    url = urllink

    video_id = extract_video_id(url)

    api = YouTubeTranscriptApi()
    fetched = api.fetch(video_id, languages=['en'])

    transcript_with_time = [
    {
        "text": snippet.text,
        "start": snippet.start
    }
    for snippet in fetched
    ]

    yttext = "\n".join(
    f"[{int(snippet.start)}s] {snippet.text}"
    for snippet in fetched
    )

print(yttext)

[0s] now stop me if you've heard this one
[1s] before but there are a lot of large
[3s] language models available today and they
[6s] have their own capabilities and
[7s] specialities what if I prefer to use one
[10s] llm to interpret some user queries in my
[12s] business application but a whole other
[15s] llm to author a response to those
[18s] queries well that scenario is exactly
[21s] what Lang chain caters to Lang chain is
[24s] an open-source orchestration framework
[27s] for the development of applications that
[29s] use large language models and it comes
[31s] in both Python and JavaScript libraries
[34s] it's it's essentially a generic
[36s] interface for nearly any llm so you have
[39s] a centralized development environment to
[41s] build your large language model
[42s] applications and then integrate them
[44s] with stuff like data sources and
[47s] software workflows now when it was
[49s] launched by Harrison Chase in October
[51s] 2022 Lang chain enjoyed a meteoric rise


In [ ]:
https://www.youtube.com/watch?v=1bUy-1hGZpI

In [79]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
tokenizer = summarizer.tokenizer
model = summarizer.model 

Device set to use cuda:0


In [80]:
tokenizer = summarizer.tokenizer
model = summarizer.model

inputs = tokenizer(
    yttext,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}
summary_ids = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=1000,
    min_length=900,
    do_sample=False
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(summary)

Lang chain is an open-source orchestration framework for the development of applications that use large language models. Lang chain streamlines the programming of llm applications through something called abstractions. It was launched by Harrison Chase in October 2022 and by June of the following year it was the single fastest growing open- source project on GitHub. There's plenty of utility here so let's take a look at its components and see what Lang chain has got to offer. It's essentially a generic interface for nearly any llm so you have a centralized development environment to build your large language model applications and then integrate them with stuff like data sources and software workflows. It also has a prompt template class that formalizes the composition of prompts and can contain instructions like do not use technical terms in your response or a set of examples to guide its responses. It can be used to create complex NLP-style applications by executing a sequence of fun

In [81]:
answer = summarizer(summary, max_length=130, min_length=30, do_sample=False)
answer

[{'summary_text': "Lang chain is an open-source orchestration framework for the development of applications that use large language models. It was launched by Harrison Chase in October 2022 and by June of the following year it was the single fastest growing open- source project on GitHub. It's essentially a generic interface for nearly any llm so you have a centralized development environment to build your applications."}]

In [82]:
print(answer[0]['summary_text'])

Lang chain is an open-source orchestration framework for the development of applications that use large language models. It was launched by Harrison Chase in October 2022 and by June of the following year it was the single fastest growing open- source project on GitHub. It's essentially a generic interface for nearly any llm so you have a centralized development environment to build your applications.


In [ ]:
"""
def extract_text(text):
    full_text  = ""
    for i in text:
        full_text += extract_text() + "\n"
    return full_text
    """

'\ndef extract_text(text):\n    full_text  = ""\n    for i in text:\n        full_text += extract_text() + "\n"\n    return full_text\n    '

In [84]:
def chunks_text(yttext , chunk_size=500 , overlap=50):
    words = yttext.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

In [86]:
def clean_text(yttext):
    text = yttext.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [87]:
def embed_chunks(chunks, model_name='sentence-transformers/all-MiniLM-L6-v2'):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, convert_to_numpy=True)
    return model, embeddings

In [88]:
def create_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index

In [89]:
def search_index(query, model, index, chunks, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]

In [ ]:
""" GENERATE_TEXT_1
def generate_text(prompt, max_length=100, num_return_sequences=1):
    inputs = qwen_tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    print(inputs["input_ids"].device)
    print(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=False,
        top_k=50,
        top_p=0.95,
    )
    return [qwen_tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
"""

In [90]:
def generate_text(prompt, max_new_tokens=300):
    inputs = qwen_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(qwen_model.device)

    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = qwen_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response

In [91]:
text = clean_text(yttext)
chunks = chunks_text(yttext,chunk_size=200, overlap=50)
model_embeddings, embeddings = embed_chunks(chunks)
index = create_faiss_index(embeddings)

In [92]:
chat_history = ""
Q = input("Enter your question :\n")
question = Q
top_chunks = search_index(question, model_embeddings, index, chunks, k=5)
context = "\n\n".join(top_chunks)
for i, chunk in enumerate(top_chunks, 1):
    print(f"\n Answer {i} \n{chunk}")


 Answer 1 
now when it was [49s] launched by Harrison Chase in October [51s] 2022 Lang chain enjoyed a meteoric rise [54s] and by June of the following year it was [56s] the single fastest growing open- source [58s] project on GitHub and while the Lang [61s] chain hype [63s] train has uh slightly cooled a little [66s] bit there's plenty of utility here so [69s] let's take a look at its components so [73s] what makes up Lang [78s] chain well Lang chain streamlines the [80s] programming of llm applications through [83s] something called abstractions now what [85s] do I mean by that well your thermostat [87s] that allows you to control the [88s] temperature in your home with without [90s] needing to understand all the complex [91s] circuitary that this entails we just set [94s] the temperature that's an abstraction so [97s] Lang chains abstractions represent [99s] common steps and Concepts necessary to [101s] work with language models and they can [104s] be chained together to create [10

In [93]:
chunk = top_chunks[:5]
context = "\n\n".join(chunk)

prompt = f"""
You are a question answering assistant.

Rules:
- Answer in one short sentence only.
- Use ONLY the provided context.
- Include the timestamp associated with the information you used.
- Copy the timestamp exactly from the context, for example: [24s].
- The timestamp is mandatory.
- Format: Answer text [timestamp]
- If the answer is not in the context, reply exactly: I don't know.

Conversation History:
{chat_history}

Context:
{context}

Question:
{question}

Answer:
"""
answer = generate_text(prompt, max_new_tokens=50)
print(answer.strip())

chat_history += f"\nUser: {question}\nAssistant: {answer}"

Lang Chain is an open-source orchestration framework for the development of applications that use large language models, coming in both Python and JavaScript libraries. [27s] It provides a generic interface for nearly any LLM, allowing users to build and integrate


In [95]:
print(chat_history)


User: What is Lang Chain ?
Assistant: Lang Chain is an open-source orchestration framework for the development of applications that use large language models, coming in both Python and JavaScript libraries. [27s] It provides a generic interface for nearly any LLM, allowing users to build and integrate
